# 🔥 FireWatch — train on a Kaggle GPU (backup to Colab)

Same stabilized RetinaNet training as the Colab notebook, but on **Kaggle** — which has a
separate free GPU quota (~30 hrs/week) and **already hosts the D-Fire dataset** (no upload).

### One-time setup (do these in the right-hand panel before running)
1. **Settings → Accelerator → GPU** (T4 or P100).
2. **Settings → Internet → On** (needed for `git clone` + `pip`).
3. **Add Data** (right panel) → search **`smoke-fire-detection-yolo`** (by *sayedgamal99*) → **Add**.

Then **Run All**. The trained model lands in `/kaggle/working/` — download it from the
**Output** panel when done (or *Save Version* to keep it).

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import tensorflow as tf
print('TensorFlow', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), 'No GPU! Settings > Accelerator > GPU, then restart.'

## 2. Get the code + install KerasHub
Needs **Internet = On**. If pip asks to restart the kernel, do it and re-run this cell.

In [ ]:
import os
os.chdir('/kaggle/working')
if not os.path.isdir('/kaggle/working/fire-detection'):
    !git clone --depth 1 -b master https://github.com/01End/fire-detection.git
os.chdir('/kaggle/working/fire-detection')
!pip install -q -U keras-hub opencv-python-headless
import keras_hub, keras
print('keras', keras.__version__, '| keras_hub', keras_hub.__version__)

## 3. Locate the D-Fire dataset (auto-detects layout under /kaggle/input)

In [ ]:
search_base = '/kaggle/input'
root = layout = None
for dp, dns, fns in os.walk(search_base):
    if os.path.isdir(os.path.join(dp, 'train', 'images')):
        root, layout = dp, 'split_first'; break
    if os.path.isdir(os.path.join(dp, 'images', 'train')):
        root, layout = dp, 'type_first'; break
assert root, f'No dataset under {search_base}. Did you Add Data (smoke-fire-detection-yolo)?'

if layout == 'split_first':
    VAL_SPLIT = 'test' if os.path.isdir(f'{root}/test/images') else 'val'
    DATA = root
else:  # YOLO images/<split> -> symlink into a writable <split>/images layout
    VAL_SPLIT = 'test' if os.path.isdir(f'{root}/images/test') else 'val'
    DATA = '/kaggle/working/data_norm'
    for sp in ['train', VAL_SPLIT]:
        os.makedirs(f'{DATA}/{sp}', exist_ok=True)
        for kind in ['images', 'labels']:
            dst = f'{DATA}/{sp}/{kind}'
            if not os.path.exists(dst):
                os.symlink(f'{root}/{kind}/{sp}', dst)

IMG = 512
OUT = '/kaggle/working/fire_retinanet_full.weights.h5'
print('DATA =', DATA, '| VAL_SPLIT =', VAL_SPLIT, '| layout =', layout)
print('train images:', len(os.listdir(f'{DATA}/train/images')),
      f'| {VAL_SPLIT}:', len(os.listdir(f'{DATA}/{VAL_SPLIT}/images')))

## 4. Train (stable) 🚀
Low LR (1e-4) + reduce-LR-on-plateau + early-stopping, best weights saved to
`/kaggle/working`. `--epochs 20` is just a ceiling; early stopping ends it when val stalls.

In [ ]:
!python -m training.tf_train \
    --data "{DATA}" \
    --train-split train \
    --val-split {VAL_SPLIT} \
    --epochs 20 \
    --batch-size 8 \
    --image-size {IMG} \
    --lr 0.0001 \
    --out "{OUT}"

## 5. Measure accuracy on the held-out test set 📊

In [ ]:
!python -m training.tf_eval \
    --model "{OUT}" \
    --data "{DATA}/{VAL_SPLIT}" \
    --image-size {IMG} --score-thr 0.5 --limit 500

## 6. See it work — annotated detections 🔥

In [ ]:
import glob, shutil
os.makedirs('/kaggle/working/val_sample', exist_ok=True)
picked = 0
for lf in sorted(glob.glob(f'{DATA}/{VAL_SPLIT}/labels/*.txt')):
    if os.path.getsize(lf) > 0:
        stem = os.path.splitext(os.path.basename(lf))[0]
        ip = f'{DATA}/{VAL_SPLIT}/images/' + stem + '.jpg'
        if os.path.exists(ip):
            shutil.copy(ip, '/kaggle/working/val_sample/'); picked += 1
    if picked >= 8:
        break

!python -m firewatch.cli detect \
    --model "{OUT}" \
    --input /kaggle/working/val_sample \
    --backend tf --arch retinanet \
    --image-size {IMG} --score-thr 0.5 \
    --out /kaggle/working/detect_out

from IPython.display import Image, display
for p in sorted(glob.glob('/kaggle/working/detect_out/*'))[:8]:
    print(p); display(Image(p))